# 03 — Operations & Communication
**Agentic X | Studio 67** — Real components only. Run from the repository root.

## 1. Objective
Implement Operations as a genuine LLM+resource-tools agent and send structured ACL messages. This notebook does not itself select the final resolution.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'core').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from core.environment import get_case
from core.communication import MessageBus
from agents.operations import OperationsAgent
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent

## 2. Communication protocol
ACL performatives: `REQUEST`, `INFORM`, `CFP`, `PROPOSE`, `ACCEPT`, `REFUSE`. The generated messages should reflect actual agent findings.

In [ ]:
case, options=get_case('SCR-001')
bus=MessageBus(case['case_id'])
request=bus.send(1,'Coordinator','Finance','REQUEST','Assess budget for '+options[0]['option_id'])
finance_eval=FinanceAgent().evaluate(case,options[0])
reply=bus.send(1,'Finance','Coordinator','INFORM',finance_eval.reason,in_reply_to=request.message_id)
assert reply.in_reply_to==request.message_id
print([m.model_dump() for m in bus.messages])

## 3. Operations Agent (real LLM)
Requires a valid model API key. Hard resource constraints are checked before the LLM is called.

In [ ]:
operations=OperationsAgent()
operations_results=[operations.evaluate(case,o) for o in options]
for result in operations_results: print(result.model_dump())

## 4. Record real Ops decisions as ACL messages
No fixed dialogue: each message includes evaluation generated by Operations.

In [ ]:
for evaluation in operations_results:
    bus.send(1,'Operations','ALL','ACCEPT' if evaluation.approved else 'REFUSE',
             f'{evaluation.option_id}: {evaluation.reason}')
for m in bus.messages: print(m.sender,'→',m.receiver,m.performative,m.content)

## 5. Validate ACL schema and traceability
Every message must be attributable to a sender and linked to the incident.

In [ ]:
assert all(msg.case_id==case['case_id'] for msg in bus.messages)
assert all(msg.message_id and msg.performative for msg in bus.messages)
assert len({msg.message_id for msg in bus.messages})==len(bus.messages)
print('ACL messages are schema-valid.')